# 🔭 Veille des modes d'intervention (MI) de la transition écologique

Ce notebook cherche sur le web les **modes d'intervention** (dispositifs publics ou privés), leurs **rapports** et leurs **évaluations**
(Cour des comptes, CGDD, I4CE, HCC, France Stratégie, OCDE…), les fait **lire par Albert API**, puis les **range dans votre Google Drive** :

```
MyDrive/veille_MI/
├── rapports/<famille>/<mode-d-intervention>/2025_CCOMPTES_EVAL_titre.pdf
├── rapports/_Nouveaux_MI/<mi-découvert>/...        ← MI absents du référentiel
├── rapports/_A_classer/...
├── catalogue_documents.csv / catalogue_modes_intervention.csv
└── veille_modes_intervention.xlsx                  ← le tableau de synthèse
```

**Avant la première utilisation** : cliquez sur 🔑 (barre de gauche) → *Ajouter un secret* → nom `ALBERT_API_KEY`, valeur = votre clé Albert, et cochez l'accès au notebook.

Ensuite : exécutez les cellules dans l'ordre avec ▶. Les documents déjà vus ne sont jamais retraités : vous pouvez relancer autant de fois que vous voulez.

In [ ]:
#@title 1️⃣ Installer l'outil et connecter Google Drive (à chaque ouverture)
GITHUB_REPO = "https://github.com/VOTRE-COMPTE/veille-mi"  #@param {type:"string"}
DOSSIER_VEILLE = "/content/drive/MyDrive/veille_MI"  #@param {type:"string"}

import os, sys, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')

CODE = "/content/veille-mi"
if os.path.exists(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", GITHUB_REPO, CODE], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)
sys.path.insert(0, CODE); os.chdir(CODE)

from veille_mi import Veille, charger_config
from veille_mi.pipeline import Options
cfg = charger_config()
cle = userdata.get("ALBERT_API_KEY")
try:
    cle_brave = userdata.get("BRAVE_API_KEY")
except Exception:
    cle_brave = None

veille = Veille(cfg, cle, DOSSIER_VEILLE, cle_brave=cle_brave)
info = veille.albert.verifier()
print("✅ Albert connecté. Référentiel :", len(veille.mis), "modes d'intervention.")
print("📁 Dossier de veille :", DOSSIER_VEILLE)

In [ ]:
#@title 📋 (Optionnel) Voir les codes des MI et des acteurs
import pandas as pd
display(pd.DataFrame([{"id": m.id, "famille": m.famille, "mode d'intervention": m.nom} for m in veille.mis]))
display(pd.DataFrame([{"code": a["code"], "nom": a["nom"], "rôle": a["role"], "zone": a["zone"]} for a in cfg["acteurs"]]))

In [ ]:
#@title 2️⃣ 🚀 LANCER LA VEILLE { display-mode: "form" }
#@markdown **Stratégies de recherche**
evaluations_du_referentiel = True  #@param {type:"boolean"}
publications_des_evaluateurs = True  #@param {type:"boolean"}
decouverte_nouveaux_MI = True  #@param {type:"boolean"}
#@markdown **Zones**
France = True  #@param {type:"boolean"}
Europe = False  #@param {type:"boolean"}
Monde = False  #@param {type:"boolean"}
#@markdown **Ciblage (vide = tout)** — codes séparés par des virgules, ex. `MI27, MI04` ou `CCOMPTES, I4CE`. Requêtes libres séparées par ` | `
modes_intervention = ""  #@param {type:"string"}
evaluateurs = ""  #@param {type:"string"}
requetes_libres = ""  #@param {type:"string"}
consigne_decouverte = ""  #@param {type:"string"}
#@markdown **Volume**
max_requetes = 30  #@param {type:"slider", min:5, max:150, step:5}
max_documents = 25  #@param {type:"slider", min:5, max:200, step:5}
indexer_dans_albert = False  #@param {type:"boolean"}

liste = lambda s: [x.strip() for x in s.replace(";", ",").split(",") if x.strip()]
options = Options(
    modes=[m for m, ok in [("referentiel", evaluations_du_referentiel),
                           ("evaluateurs", publications_des_evaluateurs),
                           ("decouverte", decouverte_nouveaux_MI)] if ok],
    zones=[z for z, ok in [("France", France), ("Europe", Europe), ("Monde", Monde)] if ok] or ["France"],
    mi_ids=liste(modes_intervention), acteurs_codes=liste(evaluateurs),
    requete_libre=requetes_libres.replace("|", "\n"), consigne_decouverte=consigne_decouverte,
    max_requetes=max_requetes, max_documents=max_documents, indexer_albert=indexer_dans_albert)
stats = veille.lancer(options)

In [ ]:
#@title 3️⃣ (Optionnel) Ajouter ma base de documents d'évaluation
#@markdown Déposez vos PDF dans un dossier Drive, indiquez son chemin, puis ▶. Ils sont lus, renommés et rangés.
DOSSIER_MA_BASE = "/content/drive/MyDrive/base_evaluations"  #@param {type:"string"}
from pathlib import Path
fichiers = sorted(p for p in Path(DOSSIER_MA_BASE).rglob("*") if p.suffix.lower() in (".pdf", ".html", ".htm"))
print(len(fichiers), "fichiers trouvés")
stats = veille.importer_fichiers(fichiers, indexer=False)

In [ ]:
#@title 4️⃣ (Optionnel) Récupérer les sources déjà citées dans le référentiel Excel
print(len(veille.urls_referentiel), "liens trouvés dans le fichier")
stats = veille.traiter_urls(veille.urls_referentiel, origine="referentiel_sources")

In [ ]:
#@title 5️⃣ Voir les résultats
cat = veille.catalogue
print(f"📚 {len(cat.docs)} documents dont {(cat.docs['type_document'] == 'evaluation').sum()} évaluations")
print("📊 Tableau complet :", cat.exporter_excel())
print("\n🆕 Nouveaux modes d'intervention candidats")
display(cat.nouveaux_mi())
print("\n🕒 Derniers documents rangés")
display(cat.docs[["doc_id", "date_document", "code_auteur", "type_document", "mi_principal",
                  "dispositif", "titre", "fichier"]].tail(20))